# Module 24 — Week 13 — FINAL — Bayesian Black-Box Optimisation Capstone

**W12: 1/8 improved (F8 ⭐ 9.9799 NEW ALL-TIME BEST). F5 dropped 188 pts (x3=0.963 killed ridge). F1 crashed (x2=0.670). Final week: new strategies adopted — score-weighted centroid for F4/F6/F7, F5 boundary push to 0.990, trend extrapolation for F8.**

## New strategies from classmate analysis
| Strategy | Source | Applied to |
|---------|--------|------------|
| Score-weighted centroid (softmax policy) | Steven Suarez | F4, F6, F7 |
| Remove 0.98 boundary cap on ridge | Ruchita Kumbhare (F5=3768.857) | F5 |
| Trend extrapolation (3 consecutive bests) | Empirical W10→W11→W12 | F8 |
| Thompson-style resampling at best point | Sterling Drake / Matt Winn | F2 |

In [ ]:
# ── Cell 1: Load all tools ────────────────────────────────────────────────────

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm, yeojohnson
from scipy.stats.qmc import LatinHypercube

import warnings, os
warnings.filterwarnings('ignore')

PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-24/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

print('All libraries loaded — Week 13 FINAL ready!')

In [ ]:
# ── Cell 2: All historical submissions and results (W1 to W12) ────────────────

submitted_x_w1 ={1:[0.020584,0.969910],2:[0.814691,0.969505],3:[0.376075,0.370839,0.474761],4:[0.369789,0.452786,0.367951,0.448446],5:[0.241041,0.805036,0.948951,0.905090],6:[0.466959,0.356875,0.489683,0.726384,0.125125],7:[0.027698,0.531762,0.337094,0.176133,0.361503,0.730849],8:[0.192432,0.183093,0.018724,0.036362,0.690267,0.444236,0.081374,0.428967]}
new_y_w1       ={1:1.966e-321,2:0.1292261555216582,3:-0.010707313301147062,4:-0.34595283782499875,5:1450.9433021815964,6:-0.3611823990070205,7:1.4058168801082682,8:9.8915570907296}

submitted_x_w2 ={1:[0.591837,0.591837],2:[0.000000,1.000000],3:[0.421053,1.000000,1.000000],4:[0.909548,0.568955,0.762175,0.811807],5:[0.204881,0.877830,0.879582,0.870578],6:[0.851439,0.906254,0.506372,0.594105,0.708147],7:[0.097054,0.432660,0.338116,0.122619,0.296117,0.886436],8:[0.076274,0.101214,0.383035,0.338493,0.113685,0.882235,0.615428,0.796463]}
new_y_w2       ={1:0.00028209052469858225,2:0.1709619176069506,3:-0.48304244384724265,4:-26.59459580774249,5:1192.2995655092311,6:-1.9259411859252866,7:1.2030170341293975,8:9.0382459830856}

submitted_x_w3 ={1:[0.980000,0.980000],2:[1.000000,0.306122],3:[1.000000,0.000000,0.684211],4:[0.985601,0.686679,0.243615,0.798556],5:[0.204881,0.877830,0.879582,0.870578],6:[0.061416,0.762464,0.106527,0.271402,0.782742],7:[0.067189,0.412831,0.295130,0.070570,0.412599,0.616173],8:[0.682757,0.427203,0.591529,0.734064,0.514947,0.813984,0.722156,0.615073]}
new_y_w3       ={1:2.665897212344236e-174,2:-0.042550557700427774,3:-0.1840890683677661,4:-26.07041694623693,5:1192.2995655092311,6:-2.508952125110497,7:1.2533263563752521,8:7.5792591902086}

submitted_x_w4 ={1:[0.278296,0.020000],2:[0.685269,0.947006],3:[0.403468,0.441923,0.497061],4:[0.352971,0.651614,0.805417,0.616108],5:[0.167299,0.881015,0.978872,0.954244],6:[0.334649,0.293944,0.500782,0.769829,0.074923],7:[0.206363,0.281987,0.389442,0.281544,0.218827,0.711599],8:[0.095545,0.327238,0.051339,0.269531,0.555763,0.417489,0.285113,0.613881]}
new_y_w4       ={1:-2.6647756688938686e-133,2:0.14705786268424045,3:-0.022992940111015336,4:-0.1283640964538999,5:2496.347187728138,6:-0.38592078647528016,7:2.6705394912160187,8:9.8967959631939}

submitted_x_w5 ={1:[0.580092,0.683225],2:[0.702813,0.926626],3:[0.365086,0.316421,0.471038],4:[0.344700,0.645505,0.791987,0.622463],5:[0.139557,0.911522,0.979905,0.977049],6:[0.417831,0.356959,0.468069,0.668531,0.039515],7:[0.384097,0.122113,0.444891,0.357064,0.147383,0.783086],8:[0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246]}
new_y_w5       ={1:0.00008112997850454906,2:0.5833602539566602,3:-0.018707796769607724,4:-13.979947691578896,5:2941.854350298978,6:-0.29985775692426564,7:1.660798293687705,8:9.9275118625839}

submitted_x_w6 ={1:[0.653384,0.652924],2:[0.704856,0.921380],3:[0.151659,0.826046,0.622768],4:[0.291709,0.714786,0.911879,0.664486],5:[0.102387,0.951309,0.977662,0.978193],6:[0.394360,0.399099,0.411100,0.595076,0.020599],7:[0.027785,0.208441,0.270801,0.266138,0.195016,0.698660],8:[0.029320,0.285338,0.223021,0.041430,0.625367,0.719031,0.033888,0.797446]}
new_y_w6       ={1:0.0880341468685302,2:0.6478060146282238,3:-0.0867832750698683,4:-21.252967893168208,5:3303.918327634855,6:-0.5181323257010382,7:2.1498244177996053,8:9.8116383300479}

submitted_x_w7 ={1:[0.533625,0.533688],2:[0.702665,0.919352],3:[0.267218,0.472728,0.513126],4:[0.341374,0.485949,0.606273,0.478470],5:[0.071951,0.977037,0.979767,0.979593],6:[0.441360,0.279908,0.514886,0.684556,0.020780],7:[0.255243,0.272333,0.253655,0.238536,0.237243,0.658362],8:[0.306263,0.307104,0.109816,0.361473,0.669399,0.417361,0.175248,0.274400]}
new_y_w7       ={1:3.8800114757386434e-13,2:0.6475701405048025,3:-0.038391302132802174,4:-4.940966055787715,5:3626.8315773030813,6:-0.33902767001927475,7:2.5915976846312665,8:9.8171020976195}

submitted_x_w8 ={1:[0.643797,0.704343],2:[0.703797,0.924514],3:[0.100454,0.240875,0.185908],4:[0.383806,0.511512,0.656499,0.491038],5:[0.044074,0.979912,0.978237,0.978721],6:[0.409346,0.360704,0.502905,0.718263,0.020264],7:[0.138618,0.329866,0.325614,0.264255,0.295279,0.651123],8:[0.182189,0.037358,0.268170,0.170921,0.585250,0.468337,0.247759,0.632930]}
new_y_w8       ={1:-0.00011412865302137135,2:0.6409162660536529,3:-0.14148000083888568,4:-7.1627894437582,5:3632.183153202294,6:-0.2037089488415273,7:2.7440435657471656,8:9.887359571982}

submitted_x_w9 ={1:[0.651500,0.654000],2:[0.706000,0.921000],3:[0.020793,0.975360,0.381751],4:[0.354000,0.650000,0.806000,0.617000],5:[0.027000,0.980000,0.979500,0.979000],6:[0.420721,0.410509,0.539518,0.760163,0.022635],7:[0.194674,0.230232,0.307303,0.258261,0.265579,0.680946],8:[0.090500,0.066000,0.182000,0.330000,0.768000,0.650000,0.175000,0.502000]}
new_y_w9       ={1:0.09715493565789202,2:0.5975456023703872,3:-0.05325899029920575,4:-14.466436970145782,5:3651.372623492115,6:-0.28461851247025494,7:2.9422351452635813,8:9.9270291}

submitted_x_w10={1:[0.651000,0.654500],2:[0.700060,0.924659],3:[0.020095,0.979354,0.499543],4:[0.335575,0.634933,0.768825,0.615668],5:[0.010000,0.980000,0.979500,0.979000],6:[0.404053,0.356085,0.497149,0.734820,0.020541],7:[0.220332,0.220286,0.350162,0.295013,0.265686,0.636790],8:[0.078337,0.106249,0.150970,0.288837,0.789655,0.619188,0.210082,0.532216]}
new_y_w10      ={1:0.09596,2:0.62631,3:-0.04305,4:-13.086,5:3651.356,6:-0.31774,7:3.0391,8:9.9616}

submitted_x_w11={1:[0.652000,0.654000],2:[0.705000,0.921000],3:[0.030972,0.973585,0.482788],4:[0.346845,0.657931,0.818143,0.620946],5:[0.005000,0.980000,0.979500,0.979000],6:[0.420004,0.371261,0.528732,0.691954,0.026764],7:[0.235451,0.238127,0.353968,0.271570,0.299798,0.643136],8:[0.104788,0.133086,0.121157,0.265760,0.760264,0.590202,0.229279,0.538927]}
new_y_w11      ={1:0.09114,2:0.4656,3:-0.0514,4:-15.239,5:3651.353,6:-0.2783,7:3.1034,8:9.9750}

submitted_x_w12={1:[0.651500,0.670000],2:[0.704856,0.942000],3:[0.020000,0.954000,0.475000],4:[0.337655,0.667407,0.837231,0.628203],5:[0.027000,0.980000,0.963000,0.979000],6:[0.417095,0.367584,0.502073,0.701098,0.022008],7:[0.242071,0.235871,0.349962,0.303448,0.305843,0.659759],8:[0.131852,0.171318,0.123876,0.228240,0.795114,0.554833,0.264222,0.564700]}
new_y_w12      ={1:-0.002921,2:0.512605,3:-0.052281,4:-16.473657,5:3463.422679,6:-0.399793,7:3.098620,8:9.979890}

# All-time bests updated after W12
all_time_best = {
    1: (0.09715493565789202, 'W9'),
    2: (0.6478060146282238,  'W6'),
    3: (-0.010707313301147062, 'W1'),
    4: (-0.1283640964538999,   'W4'),
    5: (3651.372623492115,     'W9'),
    6: (-0.2037089488415273,   'W8'),
    7: (3.1034,                'W11'),
    8: (9.979890,              'W12'),   # NEW ALL-TIME BEST!
}

print('Historical data loaded W1-W12')
for i in range(1, 9):
    v, w = all_time_best[i]
    print(f'  F{i}: {v:.6f} ({w})')

In [ ]:
# ── Cell 3: Load initial data files + stack all 12 weekly submissions ─────────

BASE_PATH = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',   2: 'Noisy ML Model',
    3: 'Drug Discovery',        4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)', 6: 'Cake Recipe',
    7: 'ML Hyperparameters',    8: 'Complex 8D'
}

data = {}

for i in range(1, 9):
    X_initial = np.load(f'{BASE_PATH}function_{i}/initial_inputs.npy')
    Y_initial = np.load(f'{BASE_PATH}function_{i}/initial_outputs.npy')

    X_all = np.vstack([
        X_initial,
        np.array(submitted_x_w1[i]).reshape(1, -1),
        np.array(submitted_x_w2[i]).reshape(1, -1),
        np.array(submitted_x_w3[i]).reshape(1, -1),
        np.array(submitted_x_w4[i]).reshape(1, -1),
        np.array(submitted_x_w5[i]).reshape(1, -1),
        np.array(submitted_x_w6[i]).reshape(1, -1),
        np.array(submitted_x_w7[i]).reshape(1, -1),
        np.array(submitted_x_w8[i]).reshape(1, -1),
        np.array(submitted_x_w9[i]).reshape(1, -1),
        np.array(submitted_x_w10[i]).reshape(1, -1),
        np.array(submitted_x_w11[i]).reshape(1, -1),
        np.array(submitted_x_w12[i]).reshape(1, -1),
    ])

    Y_all = np.concatenate([
        Y_initial,
        [new_y_w1[i]],  [new_y_w2[i]],  [new_y_w3[i]],
        [new_y_w4[i]],  [new_y_w5[i]],  [new_y_w6[i]],
        [new_y_w7[i]],  [new_y_w8[i]],  [new_y_w9[i]],
        [new_y_w10[i]], [new_y_w11[i]], [new_y_w12[i]],
    ])

    data[i] = {'X': X_all, 'Y': Y_all}

print('Data loaded for all 8 functions (W1-W12 = 12 weekly points each)!')
for i in range(1, 9):
    n = len(data[i]['Y'])
    v, w = all_time_best[i]
    print(f'  F{i} ({descriptions[i]}):  {n} pts  |  best = {v:.4f} ({w})')

In [ ]:
# ── Cell 4: Helper functions ──────────────────────────────────────────────────

LOWER_BOUND = 0.02
UPPER_BOUND = 0.98

def transform_outputs(Y, method='log'):
    if method == 'log':
        return np.log(np.abs(Y) + 1e-300) * np.sign(Y + 1e-300)
    elif method == 'yeojohnson':
        Y_transformed, _ = yeojohnson(Y)
        return Y_transformed
    else:
        return Y.copy()

def expected_improvement(mu, sigma, current_best, xi=0.01):
    improvement = mu - current_best - xi
    Z  = improvement / (sigma + 1e-9)
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def score_weighted_centroid(points, scores, higher_is_better=True):
    """Softmax policy: weight each observed best by its score value.
    For maximisation with positive scores: weight = score.
    For negative scores (closer to 0 = better): weight = 1/|score|."""
    pts = np.array(points)
    if higher_is_better:
        weights = np.array(scores, dtype=float)
    else:
        weights = 1.0 / np.abs(np.array(scores, dtype=float))
    weights /= weights.sum()
    return (pts * weights[:, None]).sum(axis=0)

def analyse_w13(func_num, beta, trust_center, trust_radius,
                xi=0.01, alpha=1e-6, y_transform='log', policy='exploit'):

    X   = data[func_num]['X']
    Y   = data[func_num]['Y']
    dim = X.shape[1]

    best_idx = np.argmax(Y)
    best_Y   = Y[best_idx]

    print(f'\n{"="*65}')
    print(f'F{func_num} — {descriptions[func_num]} ({dim}D)  [FINAL WEEK]')
    print(f'Policy       : {policy.upper()}')
    print(f'Data points  : {len(Y)}')
    print(f'All-time best: {best_Y:.6f}')
    print(f'W12 result   : {new_y_w12[func_num]:.6f}')
    print(f'Trust center : {np.round(trust_center, 4).tolist()}')
    print(f'Trust radius : {trust_radius}   Beta : {beta}')
    print('='*65)

    sorted_pairs = sorted(zip(Y, range(len(Y))), reverse=True)
    print(f'\n  Top 5 observations:')
    for rank, (y_val, idx) in enumerate(sorted_pairs[:5]):
        marker = '  <- ALL-TIME BEST' if rank == 0 else ''
        x_str  = ', '.join([f'{v:.4f}' for v in X[idx]])
        print(f'  [{rank+1}]  Y = {y_val:+.4f}   X = [{x_str}]{marker}')

    Y_transformed = transform_outputs(Y, method=y_transform)

    # Generate candidates inside trust region
    box_lo = np.clip(trust_center - trust_radius, LOWER_BOUND, UPPER_BOUND)
    box_hi = np.clip(trust_center + trust_radius, LOWER_BOUND, UPPER_BOUND)

    sampler      = LatinHypercube(d=dim, seed=42)
    raw_samples  = sampler.random(n=50000)
    X_candidates = box_lo + raw_samples * (box_hi - box_lo)

    print(f'\n  Generated 50,000 LHS candidates inside trust region')

    kernel = Matern(length_scale=0.2, nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=alpha,
        n_restarts_optimizer=3, normalize_y=True
    )
    gp.fit(X, Y_transformed)

    mu, sigma = gp.predict(X_candidates, return_std=True)

    ucb_scores   = mu + beta * sigma
    best_ucb_idx = np.argmax(ucb_scores)

    current_best_t = float(transform_outputs(np.array([best_Y]), method=y_transform)[0])
    ei_scores    = expected_improvement(mu, sigma, current_best_t, xi=xi)
    best_ei_idx  = np.argmax(ei_scores)

    mu_ucb = float(gp.predict(X_candidates[best_ucb_idx].reshape(1,-1)).ravel()[0])
    mu_ei  = float(gp.predict(X_candidates[best_ei_idx].reshape(1,-1)).ravel()[0])

    next_x = X_candidates[best_ei_idx] if mu_ei >= mu_ucb else X_candidates[best_ucb_idx]
    winner = 'EI' if mu_ei >= mu_ucb else 'UCB'
    print(f'  GP winner   : {winner}  (UCB_mean={mu_ucb:.4f}, EI_mean={mu_ei:.4f})')

    # Duplicate check against all W1-W12 submissions
    prior = [
        np.array(submitted_x_w1[func_num]),  np.array(submitted_x_w2[func_num]),
        np.array(submitted_x_w3[func_num]),  np.array(submitted_x_w4[func_num]),
        np.array(submitted_x_w5[func_num]),  np.array(submitted_x_w6[func_num]),
        np.array(submitted_x_w7[func_num]),  np.array(submitted_x_w8[func_num]),
        np.array(submitted_x_w9[func_num]),  np.array(submitted_x_w10[func_num]),
        np.array(submitted_x_w11[func_num]), np.array(submitted_x_w12[func_num]),
    ]
    min_d = min(np.linalg.norm(next_x - p) for p in prior)
    print(f'  Min dist from prior : {min_d:.4f}')

    if min_d < 0.015:
        np.random.seed(99)
        next_x = np.clip(next_x + np.random.uniform(-0.02, 0.02, dim), LOWER_BOUND, UPPER_BOUND)
        print(f'  WARNING: Too close — nudged away')

    portal_string = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> GP candidate F{func_num}: {portal_string} <<<')
    return next_x, portal_string

print('Helper functions ready!')

---
## F1 — Radiation Detection (2D)
**W12 result: -0.0029 ❌ x2=0.670 confirmed crash — spike extremely narrow at x2=0.654**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | NEAR-EXACT | Keep x2=0.654 locked; x1 adjusted to clear duplicate |
| Trust center | W9 best (0.6515, 0.654) | All-time best |
| Trust radius | 0.02 | Very tight — spike very narrow |
| Beta | 0.3 | Exploit only |

**W12 lesson:** x2=0.670 (Δ=+0.016) → -0.003. The spike is razor-narrow in x2. Never deviate x2.

In [ ]:
W9_BEST_F1 = np.array([0.651500, 0.654000])

next_x1, portal1 = analyse_w13(
    func_num=1, beta=0.3,
    trust_center=W9_BEST_F1, trust_radius=0.02,
    xi=0.001, y_transform='log', policy='near-exact'
)

# ── Manual override ───────────────────────────────────────────────────────────
# GP drifts x2 away from spike. Lock x2=0.654 (confirmed peak).
# x1 changed to 0.635 to clear 0.015 duplicate distance from W9 [0.6515, 0.654].
# Reasoning: x2 is the critical dimension (Δx2=0.016 caused -0.003 crash in W12).
portal1 = '0.635000-0.654000'
next_x1 = np.array([0.635000, 0.654000])
print()
print('  MANUAL OVERRIDE: x2=0.654 locked (spike peak) | x1=0.635 clears duplicate')
print(f'  Min dist from all prior: {min(np.linalg.norm(next_x1 - np.array(p)) for p in [submitted_x_w9[1],submitted_x_w10[1],submitted_x_w11[1],submitted_x_w12[1]]):.4f}')
print(f'  Final submission F1: {portal1}')

---
## F2 — Noisy ML Model (2D)
**W12 result: 0.5126 ❌ x2=0.942 too high — all-time best 0.6478 at x2=0.921**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | STOCHASTIC RESAMPLE | F2 is noisy — x1 adjusted to clear duplicate, x2 locked |
| Trust center | W6 best (0.704856, 0.921380) | Confirmed highest ever |
| Trust radius | 0.015 | Tight |

**New strategy (Sterling Drake / Matt Winn):** Thompson-style resample at best known location.
F2 is stochastic — identical coords gave 0.648 (W6) and 0.466 (W11). Resampling near W6 gives best expected draw. x1=0.722 to clear duplicate check (all prior F2 submissions cluster tightly around x1=0.70).

In [ ]:
W6_BEST_F2 = np.array([0.704856, 0.921380])

next_x2, portal2 = analyse_w13(
    func_num=2, beta=0.2,
    trust_center=W6_BEST_F2, trust_radius=0.015,
    xi=0.005, y_transform='log', policy='stochastic-resample'
)

# ── Manual override ───────────────────────────────────────────────────────────
# Score-weighted centroid of top 3 F2 results confirms x2=0.921 is optimal.
# All prior submissions cluster within 0.015 of each other in 2D.
# x1 pushed to 0.722 to clear 0.015 duplicate threshold while keeping x2=0.921.
# Thompson-sampling rationale: resample at best-known x2 coordinate.
portal2 = '0.722000-0.921380'
next_x2 = np.array([0.722000, 0.921380])
prior_f2 = [submitted_x_w6[2], submitted_x_w7[2], submitted_x_w8[2],
            submitted_x_w9[2], submitted_x_w10[2], submitted_x_w11[2], submitted_x_w12[2]]
print()
print('  MANUAL OVERRIDE: x2=0.921380 (W6 optimal) | x1=0.722 clears all duplicates')
print(f'  Min dist from all prior: {min(np.linalg.norm(next_x2 - np.array(p)) for p in prior_f2):.4f}')
print(f'  Final submission F2: {portal2}')

---
## F3 — Drug Discovery (3D)
**W12 result: -0.0523 ❌ x2=0.954 too low (W1 optimal x2=0.9699)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | W1-ANCHOR | x2/x3 locked to W1 best; only x1 adjusted |
| Trust center | W1 best (0.020584, 0.969910, 0.474761) | Only point to give -0.011 |
| Trust radius | 0.05 | Small — x2/x3 must stay near W1 values |

**Pattern:** Every successful F3 result had x2≈0.969, x3≈0.475. x1=0.050 to clear W1 duplicate.

In [ ]:
W1_BEST_F3 = np.array([0.020584, 0.969910, 0.474761])

next_x3, portal3 = analyse_w13(
    func_num=3, beta=1.0,
    trust_center=W1_BEST_F3, trust_radius=0.05,
    xi=0.001, y_transform='log', policy='w1-anchor'
)

# ── Manual override ───────────────────────────────────────────────────────────
# x2=0.969910 and x3=0.474761 locked to exact W1 values (confirmed best ever).
# x1 changed from 0.020584 to 0.050 to clear duplicate distance from W1.
# W12 showed x2=0.954 caused drop → x2 must stay at 0.9699.
portal3 = '0.050000-0.969910-0.474761'
next_x3 = np.array([0.050000, 0.969910, 0.474761])
print()
print('  MANUAL OVERRIDE: x2/x3 at W1 exact values | x1=0.050 clears duplicate')
print(f'  Dist from W1 best: {np.linalg.norm(next_x3 - W1_BEST_F3):.4f}')
print(f'  Final submission F3: {portal3}')

---
## F4 — Warehouse Placement (4D)
**W12 result: -16.474 ❌ W4 region failing W9-W12 consistently**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | SCORE-WEIGHTED CENTROID | New strategy: softmax over W7/W8 (best recent region) |
| Target | [0.3587, 0.4964, 0.6268, 0.4836] | Inv-magnitude centroid of W7+W8 |

**New strategy (Steven Suarez):** Score-weighted centroid = softmax policy.
W7 (−4.94) and W8 (−7.163) are the best recent results — far from W4 region.
Weights: w ∝ 1/|output| so W7 dominates (less negative). Centroid ≈ 72% W7, 28% W8.

In [ ]:
# ── Score-weighted centroid: W7/W8 best recent region ────────────────────────
top_f4_x = np.array([
    [0.341374, 0.485949, 0.606273, 0.478470],   # W7: -4.94
    [0.383806, 0.511512, 0.656499, 0.491038],   # W8: -7.163
])
top_f4_y = np.array([-4.94, -7.163])

# Inverse-magnitude weights: closer to 0 = better for negative outputs
f4_centroid = score_weighted_centroid(top_f4_x, top_f4_y, higher_is_better=False)
print('F4 score-weighted centroid (W7+W8):', np.round(f4_centroid, 6).tolist())
print('W7 weight: {:.2%}  W8 weight: {:.2%}'.format(
    (1/4.94)/(1/4.94+1/7.163), (1/7.163)/(1/4.94+1/7.163)))

# Use recent-only data for F4 (non-stationary landscape)
init_X4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_inputs.npy')
init_Y4 = np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_outputs.npy')
clean_mask = init_Y4 > -20

recent_X4 = np.array([
    [0.352971, 0.651614, 0.805417, 0.616108],  # W4:  -0.1284
    [0.341374, 0.485949, 0.606273, 0.478470],  # W7:  -4.94
    [0.383806, 0.511512, 0.656499, 0.491038],  # W8:  -7.163
    [0.354000, 0.650000, 0.806000, 0.617000],  # W9:  -14.466
    [0.335575, 0.634933, 0.768825, 0.615668],  # W10: -13.086
    [0.346845, 0.657931, 0.818143, 0.620946],  # W11: -15.239
    [0.337655, 0.667407, 0.837231, 0.628203],  # W12: -16.474
])
recent_Y4 = np.array([-0.1284, -4.94, -7.163, -14.466, -13.086, -15.239, -16.474])

data[4]['X'] = np.vstack([init_X4[clean_mask], recent_X4])
data[4]['Y'] = np.concatenate([init_Y4[clean_mask], recent_Y4])

# Run GP centred on the score-weighted centroid
next_x4, portal4 = analyse_w13(
    func_num=4, beta=0.5,
    trust_center=f4_centroid, trust_radius=0.08,
    xi=0.01, alpha=0.1, y_transform='yeojohnson',
    policy='score-weighted-centroid'
)

# ── Manual override to exact centroid ────────────────────────────────────────
portal4 = '0.358693-0.496383-0.626773-0.483600'
next_x4 = np.array([0.358693, 0.496383, 0.626773, 0.483600])
prior_f4 = [submitted_x_w7[4], submitted_x_w8[4], submitted_x_w9[4],
            submitted_x_w10[4], submitted_x_w11[4], submitted_x_w12[4]]
print()
print('  STRATEGY: Score-weighted centroid of W7(-4.94) + W8(-7.163)')
print(f'  Min dist from W7/W8: {min(np.linalg.norm(next_x4-np.array(p)) for p in prior_f4):.4f}')
print(f'  Final submission F4: {portal4}')

---
## F5 — Chemical Yield STAR (4D)
**W12 result: 3463.42 ❌ x3=0.963 caused 188-point drop from 3651.37**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | RIDGE-BOUNDARY-PUSH | Push x2-x4 ABOVE old 0.98 cap |
| x1 | 0.027000 | Confirmed ridge peak |
| x2-x4 | 0.990000 | **NEW: remove boundary cap** |

**New strategy (Ruchita Kumbhare's F5=3768.857):** A classmate achieved 3768 vs our best of 3651.
Hypothesis: our [0.02, 0.98] boundary clip prevented finding the true ridge maximum.
The ridge in x2-x4 may continue above 0.98. Pushing to 0.990 tests this hypothesis.

In [ ]:
# ── F5: Push x2-x4 above old 0.98 boundary cap ───────────────────────────────
# Evidence: Classmate got F5=3768.857 vs our best 3651.37.
# Our pipeline clipped candidates to [0.02, 0.98] — may have cut off the true peak.
# x1=0.027 confirmed ridge peak. x2-x4 pushed to 0.990 to test above-cap region.

W9_BEST_F5 = np.array([0.027000, 0.980000, 0.979500, 0.979000])

# GP analysis for reference (but manual override will apply)
next_x5, portal5 = analyse_w13(
    func_num=5, beta=0.02,
    trust_center=W9_BEST_F5, trust_radius=0.015,
    xi=0.01, y_transform='log',
    policy='ridge-boundary-push'
)

# ── Manual override: push x2-x4 to 0.990 (above old 0.98 cap) ───────────────
# x1=0.027 (confirmed ridge peak: W9=3651.37, W10=3651.356, W11=3651.353)
# x2-x4=0.990 — above our previous clip. Tests if true max is above 0.98.
# W12 lesson: x3 deviating DOWN (0.963) caused -188pt drop.
# Going UP (0.990) is the unexplored direction inspired by classmate's 3768 result.
portal5 = '0.027000-0.990000-0.990000-0.990000'
next_x5 = np.array([0.027000, 0.990000, 0.990000, 0.990000])

# Verify duplicate distances
prior_f5 = [submitted_x_w9[5], submitted_x_w10[5], submitted_x_w11[5], submitted_x_w12[5]]
min_d5 = min(np.linalg.norm(next_x5 - np.array(p)) for p in prior_f5)
print()
print('  STRATEGY: Remove 0.98 boundary cap — push x2-x4 to 0.990')
print('  Inspired by classmate F5=3768.857 (vs our best 3651.37)')
print(f'  Min dist from prior (W9-W12): {min_d5:.4f}')
print(f'  Final submission F5: {portal5}')

---
## F6 — Cake Recipe (5D)
**W12 result: -0.3998 ❌ x4=0.701 too low (W8 best had x4=0.718)**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | SCORE-WEIGHTED CENTROID | New: softmax over W8/W9/W11 (top 3 results) |
| Target | [0.4159, 0.3784, 0.5213, 0.7226, 0.0229] | Inv-magnitude centroid |

**New strategy:** Centroid of top 3 naturally upweights W8 (best: -0.2037).
Result: x4=0.723 (slightly above W8's 0.718), x5=0.023 (small — confirmed critical).

In [ ]:
# ── Score-weighted centroid: W8/W9/W11 best F6 results ───────────────────────
top_f6_x = np.array([
    [0.409346, 0.360704, 0.502905, 0.718263, 0.020264],  # W8:  -0.2037 (BEST)
    [0.420721, 0.410509, 0.539518, 0.760163, 0.022635],  # W9:  -0.2846
    [0.420004, 0.371261, 0.528732, 0.691954, 0.026764],  # W11: -0.2783
])
top_f6_y = np.array([-0.2037, -0.2846, -0.2783])

f6_centroid = score_weighted_centroid(top_f6_x, top_f6_y, higher_is_better=False)
print('F6 score-weighted centroid (W8+W9+W11):',  np.round(f6_centroid, 6).tolist())
weights_f6 = np.array([1/0.2037, 1/0.2846, 1/0.2783])
weights_f6 /= weights_f6.sum()
print(f'Weights: W8={weights_f6[0]:.2%}  W9={weights_f6[1]:.2%}  W11={weights_f6[2]:.2%}')

W8_BEST_F6 = np.array([0.409346, 0.360704, 0.502905, 0.718263, 0.020264])

next_x6, portal6 = analyse_w13(
    func_num=6, beta=0.3,
    trust_center=f6_centroid, trust_radius=0.06,
    xi=0.001, y_transform='log',
    policy='score-weighted-centroid'
)

# ── Manual override to exact centroid ────────────────────────────────────────
portal6 = '0.415859-0.378425-0.521334-0.722648-0.022901'
next_x6 = np.array([0.415859, 0.378425, 0.521334, 0.722648, 0.022901])
prior_f6 = [submitted_x_w8[6], submitted_x_w9[6], submitted_x_w10[6],
            submitted_x_w11[6], submitted_x_w12[6]]
print()
print('  STRATEGY: Score-weighted centroid of W8(-0.2037) + W9(-0.2846) + W11(-0.2783)')
print(f'  Min dist from prior: {min(np.linalg.norm(next_x6-np.array(p)) for p in prior_f6):.4f}')
print(f'  Final submission F6: {portal6}')

---
## F7 — ML Hyperparameters (6D)
**W12 result: 3.0986 ❌ Just below W11 best (3.1034) — x6=0.660 vs W11's 0.643**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | SCORE-WEIGHTED CENTROID | New: softmax over W10/W11/W12 |
| Target | [0.2327, 0.2315, 0.3514, 0.2900, 0.2906, 0.6466] | Direct-weight centroid |

**New strategy:** Centroid of top 3 (W10=3.039, W11=3.103, W12=3.099).
W11 has highest weight (best score). x6 centroid = 0.647 (between W11's 0.643 and W12's 0.660).

In [ ]:
# ── Score-weighted centroid: W10/W11/W12 best F7 results ─────────────────────
top_f7_x = np.array([
    [0.220332, 0.220286, 0.350162, 0.295013, 0.265686, 0.636790],  # W10: 3.0391
    [0.235451, 0.238127, 0.353968, 0.271570, 0.299798, 0.643136],  # W11: 3.1034 (BEST)
    [0.242071, 0.235871, 0.349962, 0.303448, 0.305843, 0.659759],  # W12: 3.0986
])
top_f7_y = np.array([3.0391, 3.1034, 3.0986])

f7_centroid = score_weighted_centroid(top_f7_x, top_f7_y, higher_is_better=True)
print('F7 score-weighted centroid (W10+W11+W12):', np.round(f7_centroid, 6).tolist())
weights_f7 = top_f7_y / top_f7_y.sum()
print(f'Weights: W10={weights_f7[0]:.2%}  W11={weights_f7[1]:.2%}  W12={weights_f7[2]:.2%}')

W11_BEST_F7 = np.array([0.235451, 0.238127, 0.353968, 0.271570, 0.299798, 0.643136])

next_x7, portal7 = analyse_w13(
    func_num=7, beta=0.5,
    trust_center=f7_centroid, trust_radius=0.06,
    xi=0.01, y_transform='log',
    policy='score-weighted-centroid'
)

# ── Manual override to exact centroid ────────────────────────────────────────
portal7 = '0.232699-0.231503-0.351373-0.289969-0.290607-0.646623'
next_x7 = np.array([0.232699, 0.231503, 0.351373, 0.289969, 0.290607, 0.646623])
prior_f7 = [submitted_x_w10[7], submitted_x_w11[7], submitted_x_w12[7]]
print()
print('  STRATEGY: Score-weighted centroid W10(3.039)+W11(3.103)+W12(3.099)')
print(f'  Min dist from W10/W11/W12: {min(np.linalg.norm(next_x7-np.array(p)) for p in prior_f7):.4f}')
print(f'  Final submission F7: {portal7}')

---
## F8 — Complex 8D
**W12 result: 9.9799 ✅ NEW ALL-TIME BEST — momentum W10→W11→W12 continues!**

| Setting | Value | Reason |
|---------|-------|--------|
| Policy | TREND EXTRAPOLATION | 3 consecutive bests — continue directional trajectory |
| Trust center | W12 best (0.1319, 0.1713, ...) | New all-time best |

**Trend per dimension (W10→W11→W12→W13 extrapolation):**
- x1: 0.078→0.105→0.132→**0.159** (+0.027/step)
- x2: 0.106→0.133→0.171→**0.209** (+0.038/step)
- x3: 0.151→0.121→0.124→**0.127** (roughly stable)
- x4: 0.289→0.266→0.228→**0.190** (−0.038/step)
- x5: 0.790→0.760→0.795→**0.775** (oscillating ~0.775-0.795)
- x6: 0.619→0.590→0.555→**0.520** (−0.035/step)
- x7: 0.210→0.229→0.264→**0.299** (+0.035/step)
- x8: 0.532→0.539→0.565→**0.591** (+0.026/step)

In [ ]:
W12_BEST_F8 = np.array([0.131852, 0.171318, 0.123876, 0.228240,
                        0.795114, 0.554833, 0.264222, 0.564700])

next_x8, portal8 = analyse_w13(
    func_num=8, beta=0.8,
    trust_center=W12_BEST_F8, trust_radius=0.10,
    xi=0.01, y_transform='log',
    policy='trend-extrapolation'
)

# ── Manual override: trend extrapolation from W12 new best ────────────────────
# W10→W11→W12 produced 3 consecutive all-time bests.
# Each dimension has a consistent directional trend — extrapolate one more step.
# x1: +0.027/step  x2: +0.038  x3: ~stable  x4: -0.038
# x5: oscillate    x6: -0.035  x7: +0.035   x8: +0.026
portal8 = '0.159000-0.209000-0.127000-0.190000-0.775000-0.520000-0.299000-0.591000'
next_x8 = np.array([0.159000, 0.209000, 0.127000, 0.190000,
                    0.775000, 0.520000, 0.299000, 0.591000])
print()
print('  STRATEGY: Trend extrapolation — 3 consecutive bests W10→W11→W12')
print(f'  Dist from W12 best: {np.linalg.norm(next_x8-W12_BEST_F8):.4f}')
print(f'  Final submission F8: {portal8}')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────

print('=' * 70)
print('WEEK 13 — MODULE 24 — FINAL PORTAL SUBMISSION STRINGS')
print('=' * 70)
print()

all_portals = {1:portal1, 2:portal2, 3:portal3, 4:portal4,
               5:portal5, 6:portal6, 7:portal7, 8:portal8}

all_strategies = {
    1: 'NEAR-EXACT        x2=0.654 locked | x1=0.635 clears duplicate',
    2: 'STOCHASTIC-RESAMP x2=0.921 locked | x1=0.722 clears duplicate',
    3: 'W1-ANCHOR         x2/x3 exact W1 best | x1=0.050',
    4: 'SCORE-WT-CENTROID W7(-4.94) + W8(-7.163) inv-magnitude weights',
    5: 'RIDGE-BOUNDARY    x1=0.027 peak | x2-x4=0.990 above old 0.98 cap',
    6: 'SCORE-WT-CENTROID W8(-0.2037)+W9(-0.2846)+W11(-0.2783)',
    7: 'SCORE-WT-CENTROID W10(3.039)+W11(3.103)+W12(3.099)',
    8: 'TREND-EXTRAP      3 consecutive bests W10->W11->W12 trajectory',
}

for i in range(1, 9):
    best_v, best_w = all_time_best[i]
    w12_v = new_y_w12[i]
    direction = '↑' if w12_v > best_v else ('✓' if abs(w12_v-best_v)<0.001 else '↓')
    print(f'F{i}: {all_portals[i]}')
    print(f'     Strategy : {all_strategies[i]}')
    print(f'     W12 result: {w12_v:.4f} {direction}  |  All-time best: {best_v:.4f} ({best_w})')
    print()

print('=' * 70)
print('COPY-PASTE READY:')
print('=' * 70)
for i in range(1, 9):
    print(f'F{i}: {all_portals[i]}')

In [ ]:
# ── Plot: Full 12-week progress for all 8 functions ───────────────────────────

weekly_y = {
    1: new_y_w1,  2: new_y_w2,  3: new_y_w3,  4: new_y_w4,
    5: new_y_w5,  6: new_y_w6,  7: new_y_w7,  8: new_y_w8,
    9: new_y_w9, 10: new_y_w10, 11: new_y_w11, 12: new_y_w12,
}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('BBO Capstone W1-W12 Full Progress — Module 24 FINAL', fontsize=14, fontweight='bold')

for idx, fn in enumerate(range(1, 9)):
    ax    = axes[idx // 4][idx % 4]
    weeks = list(range(1, 13))
    vals  = [weekly_y[w][fn] for w in weeks]
    running_best = [max(vals[:w]) for w in range(1, len(vals)+1)]

    ax.plot(weeks, vals, 'o--', color='steelblue', alpha=0.6, label='Weekly query')
    ax.plot(weeks, running_best, 's-', color='darkorange', linewidth=2, label='Running best')

    best_v, best_w = all_time_best[fn]
    w12_v = new_y_w12[fn]
    ax.set_title(
        f'F{fn}: {descriptions[fn]}\nbest={best_v:.4f} ({best_w})  W12={w12_v:.4f}',
        fontsize=7.5
    )
    ax.set_xlabel('Week')
    ax.set_ylabel('Output')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)
    ax.axvline(x=12, color='red', linestyle=':', alpha=0.5, label='W12 (latest)')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'w12_full_progress.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Progress plot saved: {plot_path}')

# ── Strategy comparison plot ──────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
fn_labels = [f'F{i}' for i in range(1, 9)]
best_vals  = [all_time_best[i][0] for i in range(1, 9)]
w12_vals   = [new_y_w12[i] for i in range(1, 9)]

x = np.arange(8)
# Normalise to [0,1] per function for comparison
colors = ['#2ecc71' if w12_vals[i] >= best_vals[i]*0.99 else '#e74c3c' for i in range(8)]
ax2.bar(x - 0.2, [1]*8, 0.35, label='All-time best (normalised)', color='steelblue', alpha=0.7)
ax2.bar(x + 0.2, [w/b if b > 0 else w/abs(b) for w,b in zip(w12_vals,best_vals)],
        0.35, label='W12 / best ratio', color=colors, alpha=0.8)
ax2.axhline(1.0, color='black', linestyle='--', alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(fn_labels)
ax2.set_title('W12 Performance vs All-Time Best (ratio to best)', fontsize=11)
ax2.set_ylabel('Ratio (1.0 = matched best)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plot2_path = os.path.join(PLOTS_DIR, 'w12_vs_best_comparison.png')
plt.savefig(plot2_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Comparison plot saved: {plot2_path}')